# Diagnostic Feature Selection

⚠️  **IMPORTANT: This notebook is for DIAGNOSTIC VALIDATION only**

**Purpose:** Validate theory-driven variable selection for PSM-DiD, NOT to generate primary model specifications.

**Key Principle:** Econometric models are specified from KNOWLEDGE (causal theory), not from DATA (statistical filters).

**Workflow:**
1. Define your theory-driven specification
2. Run diagnostic validation:
   - Check multicollinearity (VIF)
   - Compare with auto-selected features
   - Verify data quality
3. Use theory-driven specification in PSM-DiD (NOT selected_features)
4. (Optional) Robustness check with auto-selected features

**Diagnostic Methods:**
1. Multicollinearity diagnostic (VIF for your variables)
2. Specification validation (overlap, importance, quality)
3. Specification comparison (theory-driven vs auto-selected)

In [1]:
# Import and setup
from asean_green_bonds import data, config, analysis
from asean_green_bonds.data.feature_selection import (
    diagnose_multicollinearity,
    validate_specification,
    compare_specifications
)
import pandas as pd
import numpy as np

print('Loading engineered data...')
df = data.load_processed_data()
print(f'Data shape: {df.shape}')
print(f'Columns: {df.shape[1]}')


Loading engineered data...
Data shape: (23284, 173)
Columns: 173


In [2]:
# STEP 1: DEFINE YOUR THEORY-DRIVEN SPECIFICATION
# These variables should be chosen based on causal theory, not data

# Core control variables for PSM specification
psm_vars = [
    'Firm_Size',                    # Firm size affects issuance capacity
    'Leverage',                     # Debt levels affect market access
    'return_on_assets',             # Profitability affects financing costs
    'asset_tangibility',            # Tangible assets as collateral
    'prior_green_bonds',            # Track record signals commitment
    'issuer_track_record',          # Number of prior green bond issues
    'has_green_framework',          # Documented green bond framework
]

# DiD controls: Pre-treatment lagged characteristics for parallel trends
# Use L1_ prefix for lagged variables (1-year lag)
did_controls = psm_vars + [
    'L1_esg_score',                 # Pre-treatment environmental profile
    'L1_return_on_assets',          # Pre-treatment performance trend
    'L1_Leverage',                  # Pre-treatment financial structure
]

# Verify variables exist in data
available_psm = [v for v in psm_vars if v in df.columns]
available_did = [v for v in did_controls if v in df.columns]

print(f'PSM variables available: {len(available_psm)}/{len(psm_vars)}')
missing_psm = set(psm_vars) - set(available_psm)
if missing_psm:
    print(f'Missing: {missing_psm}')
print(f'\nDiD controls available: {len(available_did)}/{len(did_controls)}')
missing_did = set(did_controls) - set(available_did)
if missing_did:
    print(f'Missing: {missing_did}')
else:
    print(f'✓ All DiD controls available!')

PSM variables available: 7/7

DiD controls available: 10/10
✓ All DiD controls available!


In [3]:
# STEP 2: VALIDATE YOUR PSM SPECIFICATION WITH DIAGNOSTICS

print('='*70)
print('DIAGNOSTIC VALIDATION: PSM SPECIFICATION')
print('='*70)

# Run comprehensive diagnostic
psm_report = validate_specification(
    df,
    theory_vars=available_psm,
    outcome_col='ESG_Score',
    control_cols=available_psm
)

print(psm_report)

DIAGNOSTIC VALIDATION: PSM SPECIFICATION

DIAGNOSTIC SPECIFICATION REPORT

OVERLAP ANALYSIS:
{'theory_vars': 7, 'auto_selected': 7, 'overlap': 7, 'overlap_pct': 100.0, 'missing_from_auto': [], 'extra_in_auto': []}

MULTICOLLINEARITY (VIF):
              Variable       VIF              Status
3  issuer_track_record  6.437843  ⚠️  Warning (5-10)
1    prior_green_bonds  4.105652                ✓ OK
2  has_green_framework  2.290145                ✓ OK
0    asset_tangibility  1.001939                ✓ OK

VARIABLE IMPORTANCE:
Empty DataFrame
Columns: []
Index: []

DATA QUALITY:
{'total_vars': 7, 'missing_pct': {'return_on_assets': np.float64(6.686995361621714), 'asset_tangibility': np.float64(0.0), 'prior_green_bonds': np.float64(0.0), 'has_green_framework': np.float64(0.0), 'Leverage': np.float64(0.09448548359388421), 'issuer_track_record': np.float64(0.0), 'Firm_Size': np.float64(0.09448548359388421)}, 'zero_variance_vars': []}

RECOMMENDATIONS:
  ✓ All theory variables selected by automa

In [4]:
# STEP 3: CHECK MULTICOLLINEARITY FOR YOUR SPECIFICATION

print('\n' + '='*70)
print('MULTICOLLINEARITY CHECK: VIF FOR PSM VARIABLES')
print('='*70)

vif_psm = diagnose_multicollinearity(df, available_psm)
print(vif_psm.to_string())

print('\nInterpretation:')
print('  ✓ OK: VIF < 5')
print('  ⚠️  Warning: VIF 5-10')
print('  ❌ High: VIF > 10')

high_vif = vif_psm[vif_psm['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\n⚠️  WARNING: {len(high_vif)} variables have VIF > 10:')
    print(high_vif[['Variable', 'VIF', 'Status']])
else:
    print('\n✅ No multicollinearity issues detected')


MULTICOLLINEARITY CHECK: VIF FOR PSM VARIABLES
              Variable       VIF              Status
2  issuer_track_record  6.437843  ⚠️  Warning (5-10)
1    prior_green_bonds  4.105652                ✓ OK
3  has_green_framework  2.290145                ✓ OK
0    asset_tangibility  1.001939                ✓ OK

Interpretation:
  ✓ OK: VIF < 5
  ⚠️  Warning: VIF 5-10
  ❌ High: VIF > 10

✅ No multicollinearity issues detected


In [5]:
# STEP 4: COMPARE THEORY-DRIVEN VS AUTO-SELECTED FEATURES
# This shows how your specification aligns with data patterns

print('\n' + '='*70)
print('SPECIFICATION COMPARISON')
print('='*70)

comparison = compare_specifications(df, available_psm, 'ESG_Score')
print(comparison.to_string())

print('\nInterpretation:')
print('  ✓ Your variables in auto-selected = Good data-theory alignment')
print('  ⚠️  Your variables NOT in auto-selected = Low-signal but essential confounders')
print('  📊 Extra auto-selected vars = Potential robustness check candidates')


SPECIFICATION COMPARISON
              Variable In_Theory_Spec In_Auto_Selected  Correlation
0            Firm_Size              ✓                ✓          NaN
1             Leverage              ✓                ✓          NaN
2    asset_tangibility              ✓                ✓          NaN
3  has_green_framework              ✓                ✓          NaN
4  issuer_track_record              ✓                ✓          NaN
5    prior_green_bonds              ✓                ✓          NaN
6     return_on_assets              ✓                ✓          NaN

Interpretation:
  ✓ Your variables in auto-selected = Good data-theory alignment
  ⚠️  Your variables NOT in auto-selected = Low-signal but essential confounders
  📊 Extra auto-selected vars = Potential robustness check candidates


In [6]:
# STEP 5: (OPTIONAL) NEW ROBUSTNESS MODULE INTEGRATION

print('\n' + '='*70)
print('ROBUSTNESS MODULE INTEGRATION')
print('='*70)

# A) Continuous greenwashing intensity metrics (if present from processing merge)
for col in ['share_certified_proceeds', 'self_labeled_share', 'is_certified_majority']:
    print(f'  {col}: {'available' if col in df.columns else 'missing'}')

if {'share_certified_proceeds', 'self_labeled_share'}.issubset(df.columns):
    issuance_mask = df.get('green_bond_issue', 0).fillna(0) > 0
    if issuance_mask.any():
        print('\nCertification intensity summary on issuance-years:')
        print(df.loc[issuance_mask, ['share_certified_proceeds', 'self_labeled_share']].describe().to_string())

# B) Optional binned heterogeneity (continuous subgroup -> quantile bins)
required_cols = {'return_on_assets', 'green_bond_active', 'ric', 'Year'}
if required_cols.issubset(df.columns):
    hetero_var = 'is_certified_majority' if 'is_certified_majority' in df.columns else 'share_certified_proceeds'
    print(f'\nRunning heterogeneous effects using {hetero_var}...')
    hetero_results = analysis.heterogeneous_effects_analysis(
        df,
        outcome='return_on_assets',
        treatment_col='green_bond_active',
        heterogeneity_var=hetero_var,
        n_bins=2 if hetero_var == 'share_certified_proceeds' else 0,
    )
    print(f'Estimated subgroup models: {len(hetero_results)}')
    for k, v in list(hetero_results.items())[:5]:
        if 'error' in v:
            print(f'  {k}: error={v["error"]}')
        else:
            print(f'  {k}: coef={v["coefficient"]:.4f}, p={v["p_value"]:.4f}, n={v["n_obs"]}')
else:
    print('\nSkipping heterogeneity demo (required panel columns missing).')



ROBUSTNESS MODULE INTEGRATION
  share_certified_proceeds: available
  self_labeled_share: available
  is_certified_majority: available

Certification intensity summary on issuance-years:
       share_certified_proceeds  self_labeled_share
count                 23.000000           23.000000
mean                   0.260870            0.739130
std                    0.448978            0.448978
min                    0.000000            0.000000
25%                    0.000000            0.500000
50%                    0.000000            1.000000
75%                    0.500000            1.000000
max                    1.000000            1.000000

Running heterogeneous effects using is_certified_majority...
Estimated subgroup models: 2
  is_certified_majority_0.0: coef=-0.0083, p=0.2957, n=21697
  is_certified_majority_1.0: coef=-0.0330, p=0.0374, n=30


## Tiered Authenticity Scoring

The tiered authenticity system handles ESG data gaps by providing three levels of analysis:

- **Tier 1 (Complete)**: ≥2 pre and ≥2 post ESG observations → Full t-test analysis
- **Tier 2 (Partial)**: ≥1 pre and ≥1 post ESG observations → Point estimate with capped confidence
- **Tier 3 (Certification only)**: No ESG data → CBI/ICMA flags only, capped score

This approach ensures all green bonds receive an authenticity assessment, with appropriate confidence levels.

In [7]:
# ============================================================
# TIERED AUTHENTICITY SCORING
# ============================================================

from asean_green_bonds.authenticity import (
    calculate_authenticity_tiered, 
    apply_authenticity_with_fallbacks,
    compute_authenticity_score, 
    generate_authenticity_report, 
    print_authenticity_report
)
import pandas as pd

print('='*60)
print('TIERED AUTHENTICITY ANALYSIS')
print('='*60)

try:
    # Load green bonds authenticated data (contains both authenticity and certification)
    auth_df = pd.read_csv('data/green_bonds_authenticated.csv')
    print(f'\nLoaded green bonds authenticated data: {len(auth_df)} bonds')
    
    # Certification data is already present in this file
    try:
        cert_df = auth_df
        cert_cols = ['Deal PermID', 'is_cbi_certified', 'is_icma_certified', 'icma_confidence']
        cert_cols = [c for c in cert_cols if c in cert_df.columns]
        if len(cert_cols) > 1 and 'Deal PermID' not in auth_df.columns:  # Only merge if necessary
            # auth_df = auth_df.merge(cert_df[cert_cols], on='Deal PermID', how='left')
            pass
        print(f'  Certification data already present in authenticated file')
    except FileNotFoundError:
        print('  Note: Certification data not found, using defaults')
        auth_df['is_cbi_certified'] = 0
        auth_df['is_icma_certified'] = 0
        auth_df['icma_confidence'] = 0.0
    
    # Fill missing certification values
    for col in ['is_cbi_certified', 'is_icma_certified']:
        if col not in auth_df.columns:
            auth_df[col] = 0
        else:
            auth_df[col] = auth_df[col].fillna(0)
    if 'icma_confidence' not in auth_df.columns:
        auth_df['icma_confidence'] = 0.0
    
    # Assign tiers based on ESG data availability
    auth_df['authenticity_tier'] = 3  # Default: Certification only
    auth_df.loc[auth_df['n_pre_obs'].fillna(0) >= 1, 'authenticity_tier'] = 2  # Partial ESG
    auth_df.loc[(auth_df['n_pre_obs'].fillna(0) >= 2) & (auth_df['n_post_obs'].fillna(0) >= 2), 'authenticity_tier'] = 1  # Complete ESG
    
    tier_counts = auth_df['authenticity_tier'].value_counts().sort_index()
    print('\nTier Distribution:')
    for tier, count in tier_counts.items():
        pct = 100 * count / len(auth_df)
        tier_name = {1: 'Complete ESG', 2: 'Partial ESG', 3: 'Certification Only'}.get(tier, 'Unknown')
        print(f'  Tier {tier} ({tier_name}): {count} bonds ({pct:.1f}%)')
    
    # Compute authenticity scores
    print('\nComputing final authenticity scores...')
    scored_df = compute_authenticity_score(auth_df)
    
    # Show score statistics
    print(f'\nAuthenticity Score Statistics:')
    print(f'  Mean: {scored_df["authenticity_score"].mean():.1f}')
    print(f'  Std: {scored_df["authenticity_score"].std():.1f}')
    print(f'  Min: {scored_df["authenticity_score"].min():.1f}')
    print(f'  Max: {scored_df["authenticity_score"].max():.1f}')
    
    # Generate and print report
    report = generate_authenticity_report(scored_df)
    print_authenticity_report(report)
    
    # Compare Tier 1 vs Tier 3 scores
    t1_scores = scored_df[scored_df['authenticity_tier'] == 1]['authenticity_score']
    t3_scores = scored_df[scored_df['authenticity_tier'] == 3]['authenticity_score']
    if len(t1_scores) > 0 and len(t3_scores) > 0:
        print(f'\nScore Comparison by Tier:')
        print(f'  Tier 1 mean score: {t1_scores.mean():.1f}')
        print(f'  Tier 3 mean score: {t3_scores.mean():.1f}')
        print(f'  Difference: {t1_scores.mean() - t3_scores.mean():.1f} points')
        
except FileNotFoundError as e:
    print(f'Data files not found: {e}')
    print('Run data preparation notebook first.')
except Exception as e:
    print(f'Error in tiered authenticity analysis: {e}')
    import traceback
    traceback.print_exc()


TIERED AUTHENTICITY ANALYSIS

Loaded green bonds authenticated data: 333 bonds
  Certification data already present in authenticated file

Tier Distribution:
  Tier 1 (Complete ESG): 7 bonds (2.1%)
  Tier 2 (Partial ESG): 8 bonds (2.4%)
  Tier 3 (Certification Only): 318 bonds (95.5%)

Computing final authenticity scores...

Authenticity Score Statistics:
  Mean: 39.3
  Std: 7.6
  Min: 10.0
  Max: 75.0

                      AUTHENTICITY SCORE REPORT                       

Total Bonds Analyzed: 333

Score Statistics:
  Mean:                   39.26
  Median:                 40.00
  Std Dev:                 7.61
  Min:                     10.00
  Max:                     75.00

Authenticity Categories:
  High       (80-100)  :          0 (  0.0%)
  Medium     (60-79)   :          5 (  1.5%)
  Low        (40-59)   :        264 ( 79.3%)
  Unverified (<40)     :         64 ( 19.2%)


Score Comparison by Tier:
  Tier 1 mean score: 37.1
  Tier 3 mean score: 38.8
  Difference: -1.7 points


In [8]:
# Compare tiered vs strict (original) method
print('\n' + '='*60)
print('COMPARISON: TIERED vs STRICT METHOD')
print('='*60)

try:
    # Strict method (original - only Tier 1 bonds get authentic flag)
    strict_df = apply_authenticity_with_fallbacks(
        'data/esg_data.csv',
        'data/green_bonds_authenticated.csv',
        fallback='strict'
    )
    
    # Tiered method
    tiered_df = apply_authenticity_with_fallbacks(
        'data/esg_data.csv',
        'data/green_bonds_authenticated.csv',
        fallback='tiered'
    )
    
    strict_authentic = strict_df['is_authentic'].sum()
    tiered_authentic = tiered_df['is_authentic'].sum()
    
    print(f'\nBonds flagged as authentic:')
    print(f'  Strict method: {strict_authentic} ({100*strict_authentic/len(strict_df):.1f}%)')
    print(f'  Tiered method: {tiered_authentic} ({100*tiered_authentic/len(tiered_df):.1f}%)')
    print(f'\nNote: Tiered method provides authenticity assessment for all bonds,')
    print(f'      while strict method only assesses bonds with complete ESG data.')
    
except Exception as e:
    print(f'Comparison skipped: {e}')


COMPARISON: TIERED vs STRICT METHOD


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)



AUTHENTICITY SCORING RESULTS
Fallback mode: strict

Tier Distribution:
  Tier 1 (Complete): 27 (8.1%)
  Tier 3 (Certification_Only): 306 (91.9%)

Authenticity Status:
  Authentic: 6 (1.8%)
  Not Authentic/Unverified: 327 (98.2%)

✅ Saved to data/green_bonds_authentic_tiered.csv


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)



AUTHENTICITY SCORING RESULTS
Fallback mode: tiered

Tier Distribution:
  Tier 1 (Complete): 27 (8.1%)
  Tier 2 (Partial): 1 (0.3%)
  Tier 3 (Certification_Only): 305 (91.6%)

Authenticity Status:
  Authentic: 6 (1.8%)
  Not Authentic/Unverified: 327 (98.2%)

✅ Saved to data/green_bonds_authentic_tiered.csv

Bonds flagged as authentic:
  Strict method: 6 (1.8%)
  Tiered method: 6 (1.8%)

Note: Tiered method provides authenticity assessment for all bonds,
      while strict method only assesses bonds with complete ESG data.
